# Chapitre 13 · Les données : construire le corpus francophone

**Construire un LLM de zéro · Partie III « S'entraîner comme un labo »**

Au chapitre 12, ton entraînement a appris à survivre aux coupures de Colab.
Il tourne, il reprend, il journalise. Reste une question : que lui donner à
manger ? Ce notebook construit le **corpus francophone** qui nourrit le GPT du
chapitre 10. Tu vas dédupliquer, filtrer, mélanger, exactement comme un vrai
labo, mais à ton échelle et de tes mains.

**Comment travailler.** Toute la leçon est là, complète et exécutable de bout
en bout : lis, exécute, modifie pour voir. À la fin, la section **Exercices**
propose quatre défis à trous, du plus simple au plus costaud, validés par des
`assert`. Tout tourne **hors ligne et sans GPU** ; le paquet `datasets` est
optionnel.

## 0. Setup

On travaille en Python pur pour le pipeline de données (hash, ensembles,
statistiques de texte). PyTorch ne sert que pour le petit entraînement de la
section 6. Le paquet `datasets` de Hugging Face est **optionnel** : s'il est
absent ou si le réseau ne répond pas, on bascule sur le corpus des fables
embarqué dans ce notebook. Le notebook ne dépend jamais du réseau.

In [ ]:
import re
import hashlib
import random
from collections import Counter, defaultdict

random.seed(42)

# datasets est optionnel : on garde une trace de sa présence
try:
    import datasets  # noqa: F401
    HAS_DATASETS = True
except Exception:
    HAS_DATASETS = False

print('datasets disponible :', HAS_DATASETS)

## 1. Le corpus fil rouge : les trente fables

On reprend la matière première du chapitre 1 : trente fables de Jean de La
Fontaine (domaine public, Wikisource, édition 1874). C'est notre corpus
**propre de référence**. Tout le pipeline consiste à retrouver ce corpus
propre après l'avoir volontairement sali de doublons et de junk.

In [ ]:
corpus = """\
LA CIGALE ET LA FOURMI
La cigale, ayant chanté
Tout l'été,
Se trouva fort dépourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui prêter
Quelque grain pour subsister
Jusqu'à la saison nouvelle.
Je vous paierai, lui dit-elle,
Avant l'oût, foi d'animal,
Intérêt et principal.
La fourmi n'est pas prêteuse :
C'est là son moindre défaut.
Que faisiez-vous au temps chaud ?
Dit-elle à cette emprunteuse. -
Nuit et jour à tout venant
Je chantais, ne vous déplaise. -
Vous chantiez, j'en suis fort aise !
Eh bien ! dansez maintenant.


LE CORBEAU ET LE RENARD
Maître corbeau, sur un arbre perché,
Tenait en son bec un fromage.
Maître renard, par l'odeur alléché,
Lui tint à peu près ce langage :
Hé ! bonjour, monsieur du corbeau.
Que vous êtes joli ! que vous me semblez beau !
Sans mentir, si votre ramage
Se rapporte à votre plumage,
Vous êtes le phénix des hôtes de ces bois.
À ces mots le corbeau ne se sent pas de joie ;
Et, pour montrer sa belle voix,
Il ouvre un large bec, laisse tomber sa proie.
Le renard s'en saisit, et dit : Mon bon monsieur,
Apprenez que tout flatteur
Vit aux dépens de celui qui l'écoute :
Cette leçon vaut bien un fromage, sans doute.
Le corbeau, honteux et confus,
Jura, mais un peu tard, qu'on ne l'y prendrait plus.


LA GRENOUILLE QUI SE VEUT FAIRE AUSSI GROSSE QUE LE BŒUF
Une grenouille vit un bœuf
Qui lui sembla de belle taille.
Elle, qui n'était pas grosse en tout comme un œuf,
Envieuse, s'étend, et s'enfle, et se travaille
Pour égaler l'animal en grosseur ;
Disant : Regardez bien, ma sœur ;
Est-ce assez ? dites-moi ; n'y suis-je point encore ? -
Nenni. - M'y voici donc ? - Point du tout. - M'y voilà ? -
Vous n'en approchez point. La chétive pécore
S'enfla si bien qu'elle creva.
Le monde est plein de gens qui ne sont pas plus sages :
Tout bourgeois veut bâtir comme les grands seigneurs,
Tout petit prince a des ambassadeurs,
Tout marquis veut avoir des pages.


LE LOUP ET LE CHIEN
Un loup n'avait que les os et la peau,
Tant les chiens faisaient bonne garde.
Ce loup rencontre un dogue aussi puissant que beau,
Gras, poli, qui s'était fourvoyé par mégarde.
L'attaquer, le mettre en quartiers,
Sire loup l'eût fait volontiers :
Mais il fallait livrer bataille ;
Et le mâtin était de taille
À se défendre hardiment.
Le loup donc l'aborde humblement,
Entre en propos, et lui fait compliment
Sur son embonpoint, qu'il admire.
Il ne tiendra qu'à vous, beau sire,
D'être aussi gras que moi, lui repartit le chien.
Quittez les bois, vous ferez bien :
Vos pareils y sont misérables,
Cancres, hères et pauvres diables,
Dont la condition est de mourir de faim.
Car, quoi ! rien d'assuré ! point de franche lippée !
Tout à la pointe de l'épée !
Suivez-moi, vous aurez un bien meilleur destin.
Le loup reprit : Que me faudra-t-il faire ?
Presque rien, dit le chien : donner la chasse aux gens
Portants bâtons, et mendiants ;
Flatter ceux du logis, à son maître complaire ;
Moyennant quoi votre salaire
Sera force reliefs de toutes les façons,
Os de poulets, os de pigeons ;
Sans parler de mainte caresse.
Le loup déjà se forge une félicité
Qui le fait pleurer de tendresse.
Chemin faisant il vit le cou du chien pelé.
Qu'est-ce là ? lui dit-il. - Rien. - Quoi ! rien ! - Peu de chose. -
Mais encor ? - Le collier dont je suis attaché
De ce que vous voyez est peut-être la cause.
Attaché ! dit le loup : vous ne courez donc pas
Où vous voulez ? - Pas toujours ; mais qu'importe ?
Il importe si bien, que de tous vos repas
Je ne veux en aucune sorte,
Et ne voudrais pas même à ce prix d'un trésor.
Cela dit, maître loup s'enfuit, et court encor.


LA BESACE
Jupiter dit un jour : Que tout ce qui respire
S'en vienne comparaître aux pieds de ma grandeur :
Si dans son composé quelqu'un trouve à redire,
Il peut le déclarer sans peur ;
Je mettrai remède à la chose.
Venez, singe ; parlez le premier, et pour cause :
Voyez ces animaux, faites comparaison
De leurs beautés avec les vôtres.
Êtes-vous satisfait ? - Moi, dit-il ; pourquoi non ?
N'ai-je pas quatre pieds aussi bien que les autres ?
Mon portrait jusqu'ici ne m'a rien reproché :
Mais pour mon frère l'ours, on ne l'a qu'ébauché ;
Jamais, s'il me veut croire, il ne se fera peindre.
L'ours venant là-dessus, on crut qu'il s'allait plaindre.
Tant s'en faut : de sa forme il se loua très-fort ;
Glosa sur l'éléphant, dit qu'on pourrait encor
Ajouter à sa queue, ôter à ses oreilles ;
Que c'était une masse informe et sans beauté.
L'éléphant étant écouté,
Tout sage qu'il était, dit des choses pareilles :
Il jugea qu'à son appétit
Dame baleine était trop grosse.
Dame fourmi trouva le ciron trop petit,
Se croyant, pour elle, un colosse.
Jupin les renvoya s'étant censurés tous,
Du reste, contents d'eux. Mais, parmi les plus fous,
Notre espèce excella ; car, tout ce que nous sommes,
Lynx envers nos pareils, et taupes envers nous,
Nous nous pardonnons tout, et rien aux autres hommes :
On se voit d'un autre œil qu'on ne voit son prochain.
Le fabricateur souverain
Nous créa besaciers tous de même manière,
Tant ceux du temps passé que du temps d'aujourd'hui :
Il fit pour nos défauts la poche de derrière,
Et celle de devant pour les défauts d'autrui.


LE LOUP ET L'AGNEAU
La raison du plus fort est toujours la meilleure :
Nous l'allons montrer tout à l'heure.
Un agneau se désaltérait
Dans le courant d'une onde pure.
Un loup survint à jeun, qui cherchait aventure,
Et que la faim en ces lieux attirait.
Qui te rend si hardi de troubler mon breuvage ?
Dit cet animal plein de rage :
Tu seras châtié de ta témérité.
Sire, répond l'agneau, que Votre Majesté
Ne se mette pas en colère ;
Mais plutôt qu'elle considère
Que je me vas désaltérant
Dans le courant,
Plus de vingt pas au-dessous d'elle ;
Et que, par conséquent, en aucune façon
Je ne puis troubler sa boisson.
Tu la troubles ! reprit cette bête cruelle ;
Et je sais que de moi tu médis l'an passé.
Comment l'aurais-je fait, si je n'étais pas né ?
Reprit l'agneau : je tette encore ma mère. -
Si ce n'est toi, c'est donc ton frère. -
Je n'en ai point. - C'est donc quelqu'un des tiens ;
Car vous ne m'épargnez guère,
Vous, vos bergers et vos chiens.
On me l'a dit : il faut que je me venge.
Là-dessus, au fond des forêts
Le loup l'emporte, et puis le mange,
Sans autre forme de procès.


LA MORT ET LE BÛCHERON
Un pauvre bucheron, tout couvert de ramée,
Sous le faix du fagot aussi bien que des ans,
Gémissant et courbé, marchait à pas pesants,
Et tâchait de gagner sa chaumine enfumée.
Enfin, n'en pouvant plus d'effort et de douleur,
Il met bas son fagot, il songe à son malheur.
Quel plaisir a-t-il eu depuis qu'il est au monde ?
En est-il un plus pauvre en la machine ronde ?
Point de pain quelquefois, et jamais de repos :
Sa femme, ses enfants, les soldats, les impôts,
Le créancier, et la corvée,
Lui font d'un malheureux la peinture achevée.
Il appelle la Mort. Elle vient sans tarder,
Lui demande ce qu'il faut faire.
C'est, dit-il, afin de m'aider
À recharger ce bois ; tu ne tarderas guère.
Le trépas vient tout guérir ;
Mais ne bougeons d'où nous sommes :
Plutôt souffrir que mourir,
C'est la devise des hommes.


LE RENARD ET LA CIGOGNE
Compère le renard se mit un jour en frais,
Et retint à dîner commère la cigogne.
Le régal fut petit et sans beaucoup d'apprêts :
Le galant, pour toute besogne,
Avait un brouet clair ; il vivait chichement.
Ce brouet fut par lui servi sur une assiette :
La cigogne au long bec n'en put attraper miette ;
Et le drôle eut lapé le tout en un moment.
Pour se venger de cette tromperie,
À quelque temps de là la cigogne le prie.
Volontiers, lui dit-il ; car avec mes amis
Je ne fais point cérémonie.
À l'heure dite, il courut au logis
De la cigogne son hôtesse ;
Loua très-fort sa politesse ;
Trouva le dîner cuit à point :
Bon appétit surtout ; renards n'en manquent point.
Il se réjouissait à l'odeur de la viande
Mise en menus morceaux, et qu'il croyait friande.
On servit, pour l'embarrasser,
En un vase à long col et d'étroite embouchure.
Le bec de la cigogne y pouvait bien passer ;
Mais le museau du sire était d'autre mesure.
Il lui fallut à jeun retourner au logis,
Honteux comme un renard qu'une poule aurait pris,
Serrant la queue, et portant bas l'oreille.
Trompeurs, c'est pour vous que j'écris :
Attendez-vous à la pareille.


LE CHÊNE ET LE ROSEAU
Le chêne un jour dit au roseau :
Vous avez bien sujet d'accuser la nature ;
Un roitelet pour vous est un pesant fardeau :
Le moindre vent qui d'aventure
Fait rider la face de l'eau,
Vous oblige à baisser la tête ;
Cependant que mon front, au Caucase pareil,
Non content d'arrêter les rayons du soleil,
Brave l'effort de la tempête.
Tout vous est aquilon, tout me semble zéphyr.
Encor si vous naissiez à l'abri du feuillage
Dont je couvre le voisinage,
Vous n'auriez pas tant à souffrir,
Je vous défendrais de l'orage :
Mais vous naissez le plus souvent
Sur les humides bords des royaumes du vent.
La nature envers vous me semble bien injuste.
Votre compassion, lui répondit l'arbuste,
Part d'un bon naturel ; mais quittez ce souci :
Les vents me sont moins qu'à vous redoutables ;
Je plie et ne romps pas. Vous avez jusqu'ici
Contre leurs coups épouvantables
Résisté sans courber le dos ;
Mais attendons la fin. Comme il disait ces mots,
Du bout de l'horizon accourt avec furie
Le plus terrible des enfants
Que le Nord eût portés jusque-là dans ses flancs.
L'arbre tient bon ; le roseau plie.
Le vent redouble ses efforts,
Et fait si bien qu'il déracine
Celui de qui la tête au ciel était voisine,
Et dont les pieds touchaient à l'empire des morts.


LE LION ET LE RAT
Il faut, autant qu'on peut, obliger tout le monde :
On a souvent besoin d'un plus petit que soi.
De cette vérité deux fables feront foi ;
Tant la chose en preuves abonde.
Entre les pattes d'un lion
Un rat sortit de terre assez à l'étourdie.
Le roi des animaux, en cette occasion,
Montra ce qu'il était, et lui donna la vie.
Ce bienfait ne fut pas perdu.
Quelqu'un aurait-il jamais cru
Qu'un lion d'un rat eût affaire ?
Cependant il advint qu'au sortir des forêts
Ce lion fut pris dans des rets,
Dont ses rugissements ne le purent défaire.
Sire rat accourut, et fit tant par ses dents
Qu'une maille rongée emporta tout l'ouvrage.
Patience et longueur de temps
Font plus que force ni que rage.


LA COLOMBE ET LA FOURMI
L'autre exemple est tiré d'animaux plus petits.
Le long d'un clair ruisseau buvait une colombe,
Quand sur l'eau se penchant une fourmis y tombe ;
Et dans cet océan on eût vu la fourmis
S'efforcer, mais en vain, de regagner la rive.
La colombe aussitôt usa de charité :
Un brin d'herbe dans l'eau par elle étant jeté,
Ce fut un promontoire où la fourmis arrive.
Elle se sauve. Et là-dessus
Passe un certain croquant qui marchait les pieds nus :
Ce croquant, par hasard, avait une arbalète.
Dès qu'il voit l'oiseau de Vénus,
Il le croit en son pot, et déjà lui fait fête.
Tandis qu'à le tuer mon villageois s'apprête,
La fourmi le pique au talon.
Le vilain retourne la tête :
La colombe l'entend, part, et tire de long.
Le souper du croquant avec elle s'envole :
Point de pigeon pour une obole.


LE LIÈVRE ET LA TORTUE
Rien ne sert de courir ; il faut partir à point :
Le lièvre et la tortue en sont un témoignage.
Gageons, dit celle-ci, que vous n'atteindrez point
Sitôt que moi ce but. Sitôt ! êtes-vous sage ?
Repartit l'animal léger :
Ma commère, il vous faut purger
Avec quatre grains d'ellébore.
- Sage ou non, je parie encore.
Ainsi fut fait ; et de tous deux
On mit près du but les enjeux.
Savoir quoi, ce n'est pas l'affaire,
Ni de quel juge l'on convint.
Notre lièvre n'avait que quatre pas à faire ;
J'entends de ceux qu'il fait lorsque, près d'être atteint,
Il s'éloigne des chiens, les renvoie aux calendes,
Et leur fait arpenter les landes.
Ayant, dis-je, du temps de reste pour brouter,
Pour dormir, et pour écouter
D'où vient le vent, il laisse la tortue
Aller son train de sénateur.
Elle part, elle s'évertue ;
Elle se hâte avec lenteur.
Lui cependant méprise une telle victoire,
Tient la gageure à peu de gloire,
Croit qu'il y va de son honneur
De partir tard. Il broute, il se repose ;
Il s'amuse à toute autre chose
Qu'à la gageure. À la fin, quand il vit
Que l'autre touchait presque au bout de la carrière,
Il partit comme un trait ; mais les élans qu'il fit
Furent vains : la tortue arriva la première.
Eh bien ! lui cria-t-elle, avais-je pas raison ?
De quoi vous sert votre vitesse ?
Moi l'emporter ! et que serait-ce
Si vous portiez une maison ?


LE RENARD ET LES RAISINS
Certain renard gascon, d'autres disent normand,
Mourant presque de faim, vit au haut d'une treille
Des raisins, mûrs apparemment,
Et couverts d'une peau vermeille.
Le galant en eût fait volontiers un repas ;
Mais comme il n'y pouvait atteindre :
Ils sont trop verts, dit-il, et bons pour des goujats.
Fit-il pas mieux que de se plaindre ?


LE HÉRON
Un jour, sur ses longs pieds, allait je ne sais où
Le héron au long bec emmanché d'un long cou :
Il côtoyait une rivière.
L'onde était transparente ainsi qu'aux plus beaux jours ;
Ma commère la carpe y faisait mille tours
Avec le brochet son compère.
Le héron en eût fait aisément son profit :
Tous approchaient du bord ; l'oiseau n'avait qu'à prendre.
Mais il crut mieux faire d'attendre
Qu'il eût un peu plus d'appétit :
Il vivait de régime, et mangeait à ses heures.
Après quelques moments l'appétit vint : l'oiseau,
S'approchant du bord, vit sur l'eau
Des tanches qui sortaient du fond de ces demeures.
Le mets ne lui plut pas, il s'attendait à mieux,
Et montrait un goût dédaigneux
Comme le rat du bon Horace.
Moi, des tanches ! dit-il ; moi, héron, que je fasse
Une si pauvre chère ! Et pour qui me prend-on ?
La tanche rebutée, il trouva du goujon.
Du goujon ! c'est bien là le dîner d'un héron !
J'ouvrirais pour si peu le bec ! aux dieux ne plaise !
Il l'ouvrit pour bien moins : tout alla de façon
Qu'il ne vit plus aucun poisson.
La faim le prit : il fut tout heureux et tout aise
De rencontrer un limaçon.
Ne soyons pas si difficiles :
Les plus accommodants, ce sont les plus habiles ;
On hasarde de perdre en voulant trop gagner.
Gardez-vous de rien dédaigner,
Surtout quand vous avez à peu près votre compte.
Bien des gens y sont pris. Ce n'est pas aux hérons
Que je parle : écoutez, humains, un autre conte :
Vous verrez que chez vous j'ai puisé ces leçons.


LA LAITIÈRE ET LE POT AU LAIT
Perrette, sur sa tête ayant un pot au lait
Bien posé sur un coussinet,
Prétendait arriver sans encombre à la ville.
Légère et court vêtue, elle allait à grands pas,
Ayant mis ce jour-là, pour être plus agile,
Cotillon simple et souliers plats.
Notre laitière ainsi troussée
Comptait déjà dans sa pensée
Tout le prix de son lait ; en employait l'argent ;
Achetait un cent d'œufs ; faisait triple couvée :
La chose allait à bien par son soin diligent.
Il m'est, disait-elle, facile
D'élever des poulets autour de ma maison ;
Le renard sera bien habile
S'il ne m'en laisse assez pour avoir un cochon.
Le porc à s'engraisser coûtera peu de son ;
Il était, quand je l'eus, de grosseur raisonnable :
J'aurai, le revendant, de l'argent bel et bon.
Et qui m'empêchera de mettre en notre étable,
Vu le prix dont il est, une vache et son veau,
Que je verrai sauter au milieu du troupeau ?
Perrette là-dessus saute aussi, transportée :
Le lait tombe ; adieu veau, vache, cochon, couvée.
La dame de ces biens, quittant d'un œil marri
Sa fortune ainsi répandue,
Va s'excuser à son mari,
En grand danger d'être battue.
Le récit en farce en fut fait ;
On l'appela le Pot au lait.
Quel esprit ne bat la campagne ?
Qui ne fait châteaux en Espagne ?
Picrochole, Pyrrhus, la laitière, enfin tous,
Autant les sages que les fous.
Chacun songe en veillant ; il n'est rien de plus doux
Une flatteuse erreur emporte alors nos âmes ;
Tout le bien du monde est à nous,
Tous les honneurs, toutes les femmes.
Quand je suis seul, je fais au plus brave un défi ;
Je m'écarte, je vais détrôner le sophi ;
On m'élit roi, mon peuple m'aime ;
Les diadèmes vont sur ma tête pleuvant :
Quelque accident fait-il que je rentre en moi-même ;
Je suis Gros-Jean comme devant.


LE COCHE ET LA MOUCHE
Dans un chemin montant, sablonneux, malaisé,
Et de tous les côtés au soleil exposé,
Six forts chevaux tiraient un coche.
Femmes, moine, vieillards, tout était descendu :
L'attelage suait, soufflait, était rendu.
Une mouche survient, et des chevaux s'approche,
Prétend les animer par son bourdonnement,
Pique l'un, pique l'autre, et pense à tout moment
Qu'elle fait aller la machine,
S'assied sur le timon, sur le nez du cocher.
Aussitôt que le char chemine,
Et qu'elle voit les gens marcher,
Elle s'en attribue uniquement la gloire,
Va, vient, fait l'empressée : il semble que ce soit
Un sergent de bataille allant en chaque endroit
Faire avancer ses gens et hâter la victoire.
La mouche, en ce commun besoin,
Se plaint qu'elle agit seule, et qu'elle a tout le soin ;
Qu'aucun n'aide aux chevaux à se tirer d'affaire.
Le moine disait son bréviaire :
Il prenait bien son temps ! une femme chantait :
C'était bien de chansons qu'alors il s'agissait !
Dame mouche s'en va chanter à leurs oreilles,
Et fait cent sottises pareilles.
Après bien du travail, le coche arrive au haut.
Respirons maintenant ! dit la mouche aussitôt :
J'ai tant fait que nos gens sont enfin dans la plaine.
Çà, messieurs les chevaux, payez-moi de ma peine.
Ainsi certaines gens, faisant les empressés,
S'introduisent dans les affaires :
Ils font partout les nécessaires,
Et, partout importuns, devraient être chassés.


LE SAVETIER ET LE FINANCIER
Un savetier chantait du matin jusqu'au soir :
C'était merveille de le voir,
Merveille de l'ouïr ; il faisait des passages :
Plus content qu'aucun des sept sages.
Son voisin, au contraire, étant tout cousu d'or,
Chantait peu, dormait moins encor :
C'était un homme de finance.
Si sur le point du jour parfois il sommeillait,
Le savetier alors en chantant l'éveillait ;
Et le financier se plaignait
Que les soins de la Providence
N'eussent pas au marché fait vendre le dormir,
Comme le manger et le boire.
En son hôtel il fait venir
Le chanteur, et lui dit : Or çà, sire Grégoire,
Que gagnez-vous par an ? Par an ! ma foi, monsieur
Dit avec un ton de rieur
Le gaillard savetier, ce n'est point ma manière
De compter de la sorte ; et je n'entasse guère
Un jour sur l'autre : il suffit qu'à la fin
J'attrape le bout de l'année ;
Chaque jour amène son pain. -
Eh bien ! que gagnez-vous, dites-moi, par journée ?
Tantôt plus, tantôt moins : le mal est que toujours
(Et sans cela nos gains seraient assez honnêtes),
Le mal est que dans l'an s'entremêlent des jours
Qu'il faut chômer ; on nous ruine en fêtes :
L'une fait tort à l'autre ; et monsieur le curé
De quelque nouveau saint charge toujours son prône.
Le financier, riant de sa naïveté,
Lui dit : Je vous veux mettre aujourd'hui sur le trône.
Prenez ces cent écus ; gardez-les avec soin,
Pour vous en servir au besoin.
Le savetier crut voir tout l'argent que la terre
Avait depuis plus de cent ans,
Produit pour l'usage des gens.
Il retourne chez lui : dans sa cave il enserre
L'argent, et sa joie à la fois.
Plus de chant : il perdit la voix
Du moment qu'il gagna ce qui cause nos peines.
Le sommeil quitta son logis :
Il eut pour hôtes les soucis,
Les soupçons, les alarmes vaines.
Tout le jour il avait l'œil au guet ; et la nuit,
Si quelque chat faisait du bruit,
Le chat prenait l'argent. À la fin le pauvre homme
S'en courut chez celui qu'il ne réveillait plus :
Rendez-moi, lui dit-il, mes chansons et mon somme ;
Et reprenez vos cent écus.


LES ANIMAUX MALADES DE LA PESTE
Un mal qui répand la terreur,
Mal que le ciel en sa fureur
Inventa pour punir les crimes de la terre,
La peste (puisqu'il faut l'appeler par son nom),
Capable d'enrichir en un jour l'Achéron,
Faisait aux animaux la guerre.
Ils ne mouraient pas tous, mais tous étaient frappés :
On n'en voyait point d'occupés
À chercher le soutien d'une mourante vie ;
Nul mets n'excitait leur envie ;
Ni loups ni renards n'épiaient
La douce et l'innocente proie ;
Les tourterelles se fuyaient :
Plus d'amour, partant plus de joie.
Le lion tint conseil, et dit : Mes chers amis,
Je crois que le ciel a permis
Pour nos péchés cette infortune.
Que le plus coupable de nous
Se sacrifie aux traits du céleste courroux ;
Peut-être il obtiendra la guérison commune.
L'histoire nous apprend qu'en de tels accidents
On fait de pareils dévouements.
Ne nous flattons donc point ; voyons sans indulgence
L'état de notre conscience.
Pour moi, satisfaisant mes appétits gloutons,
J'ai dévoré force moutons.
Que m'avaient-ils fait ? nulle offense ;
Même il m'est arrivé quelquefois de manger
Le berger.
Je me dévouerai donc, s'il le faut : mais je pense
Qu'il est bon que chacun s'accuse ainsi que moi ;
Car on doit souhaiter, selon toute justice,
Que le plus coupable périsse.
Sire, dit le renard, vous êtes trop bon roi ;
Vos scrupules font voir trop de délicatesse.
Eh bien ! manger moutons, canaille, sotte espèce,
Est-ce un péché ? Non, non. Vous leur fîtes, seigneur,
En les croquant, beaucoup d'honneur ;
Et quant au berger, l'on peut dire
Qu'il était digne de tous maux,
Étant de ces gens-là qui sur les animaux
Se font un chimérique empire.
Ainsi dit le renard ; et flatteurs d'applaudir.
On n'osa trop approfondir
Du tigre, ni de l'ours, ni des autres puissances,
Les moins pardonnables offenses :
Tous les gens querelleurs, jusqu'aux simples mâtins,
Au dire de chacun, étaient de petits saints.
L'âne vint à son tour, et dit : J'ai souvenance
Qu'en un pré de moines passant,
La faim, l'occasion, l'herbe tendre, et, je pense,
Quelque diable aussi me poussant,
Je tondis de ce pré la largeur de ma langue ;
Je n'en avais nul droit, puisqu'il faut parler net.
À ces mots, on cria haro sur le baudet.
Un loup, quelque peu clerc, prouva par sa harangue
Qu'il fallait dévouer ce maudit animal,
Ce pelé, ce galeux, d'où venait tout leur mal.
Sa peccadille fut jugée un cas pendable.
Manger l'herbe d'autrui ! quel crime abominable !
Rien que la mort n'était capable
D'expier son forfait. On le lui fit bien voir.
Selon que vous serez puissant ou misérable,
Les jugements de cour vous rendront blanc ou noir.


LA POULE AUX ŒUFS D'OR
L'avarice perd tout en voulant tout gagner.
Je ne veux, pour le témoigner,
Que celui dont la poule, à ce que dit la fable,
Pondait tous les jours un œuf d'or.
Il crut que, dans son corps, elle avait un trésor ;
Il la tua, l'ouvrit, et la trouva semblable
À celle dont les œufs ne lui rapportaient rien,
S'étant lui-même ôté le plus beau de son bien.
Belle leçon pour les gens chiches !
Pendant ces derniers temps combien en a-t-on vus
Qui du soir au matin sont pauvres devenus
Pour vouloir trop tôt être riches !


L'OURS ET LES DEUX COMPAGNONS
Deux compagnons, pressés d'argent,
À leur voisin fourreur vendirent
La peau d'un ours encor vivant,
Mais qu'ils tueraient bientôt, du moins à ce qu'ils dirent,
C'était le roi des ours au compte de ces gens.
Le marchand à sa peau devait faire fortune ;
Elle garantirait des froids les plus cuisants ;
On en pourrait fourrer plutôt deux robes qu'une.
Dindenaut prisait moins ses moutons qu'eux leur ours :
Leur, à leur compte, et non à celui de la bête.
S'offrant de la livrer au plus tard dans deux jours,
Ils conviennent de prix, et se mettent en quête,
Trouvent l'ours qui s'avance et vient vers eux au trot,
Voilà mes gens frappés comme d'un coup de foudre.
Le marché ne tint pas ; il fallut le résoudre :
D'intérêts contre l'ours, on n'en dit pas un mot.
L'un des deux compagnons grimpe au faîte d'un arbre ;
L'autre, plus froid que n'est un marbre,
Se couche sur le nez, fait le mort, tient son vent,
Ayant quelque part ouï dire
Que l'ours s'acharne peu souvent
Sur un corps qui ne vit, ne meut, ni ne respire.
Seigneur ours, comme un sot, donna dans ce panneau :
Il voit ce corps gisant, le croit privé de vie ;
Et, de peur de supercherie,
Le tourne, le retourne, approche son museau,
Flaire aux passages de l'haleine.
C'est, dit-il, un cadavre ; ôtons-nous, car il sent.
À ces mots, l'ours s'en va dans la forêt prochaine.
L'un de nos deux marchands de son arbre descend,
Court à son compagnon, lui dit que c'est merveille
Qu'il n'ait eu seulement que la peur pour tout mal.
Eh bien ! ajouta-t-il, la peau de l'animal ?
Mais que t'a-t-il dit à l'oreille ?
Car il t'approchait de bien près,
Te retournant avec sa serre.
Il m'a dit qu'il ne faut jamais
Vendre la peau de l'ours qu'on ne l'ait mis par terre.


LE RENARD ET LE BOUC
Capitaine renard allait de compagnie
Avec son ami bouc des plus haut encornés :
Celui-ci ne voyait pas plus loin que son nez ;
L'autre était passé maître en fait de tromperie.
La soif les obligea de descendre en un puits ;
Là chacun d'eux se désaltère.
Après qu'abondamment tous deux en eurent pris,
Le renard dit au bouc : Que ferons-nous, compère ?
Ce n'est pas tout de boire, il faut sortir d'ici.
Lève tes pieds en haut, et tes cornes aussi ;
Mets-les contre le mur : le long de ton échine
Je grimperai premièrement ;
Puis sur tes cornes m'élevant,
À l'aide de cette machine,
De ce lieu-ci je sortirai,
Après quoi je t'en tirerai.
Par ma barbe, dit l'autre, il est bon ; et je loue
Les gens bien sensés comme toi.
Je n'aurais jamais, quant à moi,
Trouvé ce secret, je l'avoue.
Le renard sort du puits, laisse son compagnon,
Et vous lui fait un beau sermon
Pour l'exhorter à patience.
Si le ciel t'eût, dit-il, donné par excellence
Autant de jugement que de barbe au menton,
Tu n'aurais pas, à la légère,
Descendu dans ce puits. Or, adieu ; j'en suis hors :
Tâche de t'en tirer et fais tous les efforts ;
Car, pour moi, j'ai certaine affaire
Qui ne me permet pas d'arrêter en chemin.
En toute chose il faut considérer la fin.


LE CERF SE VOYANT DANS L'EAU
Dans le cristal d'une fontaine
Un cerf se mirant autrefois
Louait la beauté de son bois,
Et ne pouvait qu'avecque peine
Souffrir ses jambes de fuseaux,
Dont il voyait l'objet se perdre dans les eaux.
Quelle proportion de mes pieds à ma tête !
Disait-il en voyant leur ombre avec douleur :
Des taillis les plus hauts mon front atteint le faîte ;
Mes pieds ne me font point d'honneur.
Tout en parlant de la sorte,
Un limier le fait partir.
Il tâche à se garantir ;
Dans les forêts il s'emporte :
Son bois, dommageable ornement,
L'arrêtant à chaque moment,
Nuit à l'office que lui rendent
Ses pieds de qui ses jours dépendent.
Il se dédit alors, et maudit les présents
Que le ciel lui fait tous les ans.
Nous faisons cas du beau, nous méprisons l'utile ;
Et le beau souvent nous détruit.
Ce cerf blâme ses pieds qui le rendent agile ;
Il estime un bois qui lui nuit.


LE LOUP DEVENU BERGER
Un loup qui commençait d'avoir petite part
Aux brebis de son voisinage,
Crut qu'il fallait s'aider de la peau du renard,
Et faire un nouveau personnage
Il s'habille en berger, endosse un hoqueton,
Fait sa houlette d'un bâton,
Sans oublier la cornemuse.
Pour pousser jusqu'au bout la ruse,
Il aurait volontiers écrit sur son chapeau :
"C'est moi qui suis Guillot, berger de ce troupeau."
Sa personne étant ainsi faite,
Et ses pieds de devant posés sur sa houlette,
Guillot le sycophante approche doucement.
Guillot, le vrai Guillot, étendu sur l'herbette,
Dormait alors profondément ;
Son chien dormait aussi, comme aussi sa musette :
La plupart des brebis dormaient pareillement.
L'hypocrite les laissa faire ;
Et, pour pouvoir mener vers son fort les brebis,
Il voulut ajouter la parole aux habits,
Chose qu'il croyait nécessaire ;
Mais cela gâta son affaire :
Il ne put du pasteur contrefaire la voix.
Le ton dont il parla fit retentir les bois,
Et découvrit tout le mystère.
Chacun se réveille à ce son,
Les brebis, le chien, le garçon.
Le pauvre loup, dans cet esclandre,
Empêché par son hoqueton,
Ne put ni fuir ni se défendre.
Toujours par quelque endroit fourbes se laissent prendre
Quiconque est loup agisse en loup ;
C'est le plus certain de beaucoup.


LE RAT DE VILLE, ET LE RAT DES CHAMPS
Autrefois le rat de ville
Invita le rat des champs,
D'une façon fort civile,
À des reliefs d'ortolans.
Sur un tapis de Turquie
Le couvert se trouva mis.
Je laisse à penser la vie
Que firent ces deux amis.
Le régal fut fort honnête ;
Rien ne manquait au festin :
Mais quelqu'un troubla la fête
Pendant qu'ils étaient en train.
À la porte de la salle
Ils entendirent du bruit :
Le rat de ville détale ;
Son camarade le suit.
Le bruit cesse, on se retire :
Rats en campagne aussitôt ;
Et le citadin de dire :
Achevons tout notre rôt.
C'est assez, dit le rustique :
Demain vous viendrez chez moi.
Ce n'est pas que je me pique
De tous vos festins de roi :
Mais rien ne vient m'interrompre ;
Je mange tout à loisir.
Adieu donc. Fi du plaisir
Que la crainte peut corrompre !


LE PETIT POISSON ET LE PÊCHEUR
Petit poisson deviendra grand,
Pourvu que Dieu lui prête vie ;
Mais le lâcher en attendant,
Je tiens pour moi que c'est folie,
Car de le rattraper il n'est pas trop certain.
Un carpeau qui n'était encore que fretin,
Fut pris par un pêcheur au bord d'une rivière.
Tout fait nombre, dit l'homme, en voyant son butin ;
Voilà commencement de chère et de festin :
Mettons-le en notre gibecière.
Le pauvre carpillon lui dit en sa manière :
Que ferez-vous de moi ? je ne saurais fournir
Au plus qu'une demi-bouchée.
Laissez-moi carpe devenir :
Je serai par vous repêchée ;
Quelque gros partisan m'achètera bien cher :
Au lieu qu'il vous en faut chercher
Peut-être encor cent de ma taille
Pour faire un plat : quel plat ! croyez-moi, rien qui vaille.
Rien qui vaille ! eh bien ! soit, repartit le pêcheur ;
Poisson, mon bel ami, qui faites le prêcheur,
Vous irez dans la poêle ; et, vous avez beau dire,
Dès ce soir on vous fera frire.
Un Tiens, vaut, ce dit-on, mieux que deux Tu l'auras :
L'un est sûr, l'autre ne l'est pas.


LE POT DE TERRE ET LE POT DE FER
Le pot de fer proposa
Au pot de terre un voyage.
Celui-ci s'en excusa,
Disant qu'il ferait que sage
De garder le coin du feu :
Car il lui fallait si peu,
Si peu que la moindre chose
De son débris serait cause :
Il n'en reviendrait morceau.
Pour vous, dit-il, dont la peau
Est plus dure que la mienne,
Je ne vois rien qui vous tienne.
Nous vous mettrons à couvert,
Repartit le pot de fer :
Si quelque matière dure
Vous menace d'aventure,
Entre deux je passerai,
Et du coup vous sauverai.
Cette offre le persuade.
Pot de fer son camarade
Se met droit à ses côtés.
Mes gens s'en vont à trois pieds
Clopin clopant comme ils peuvent,
L'un contre l'autre jetés
Au moindre hoquet qu'ils treuvent.
Le pot de terre en souffre ; il n'eut pas fait cent pas
Que par son compagnon il fut mis en éclats,
Sans qu'il eût lieu de se plaindre.
Ne nous associons qu'avecque nos égaux ;
Ou bien il nous faudra craindre
Le destin d'un de ces pots.


LE LABOUREUR ET SES ENFANTS
Travaillez, prenez de la peine :
C'est le fonds qui manque le moins.
Un riche laboureur, sentant sa mort prochaine,
Fit venir ses enfants, leur parla sans témoins.
Gardez-vous, leur dit-il, de vendre l'héritage
Que nous ont laissé nos parents :
Un trésor est caché dedans.
Je ne sais pas l'endroit ; mais un peu de courage
Vous le fera trouver : vous en viendrez à bout.
Remuez votre champ dès qu'on aura fait l'oût :
Creusez, fouillez, bêchez ; ne laissez nulle place
Où la main ne passe et repasse.
Le père mort, les fils vous retournent le champ,
De çà, de là, partout ; si bien qu'au bout de l'an
Il en rapporta davantage.
D'argent, point de caché. Mais le père fut sage
De leur montrer, avant sa mort,
Que le travail est un trésor.


LE MEUNIER, SON FILS, ET L'ÂNE
L'invention des arts étant un droit d'aînesse,
Nous devons l'apologue à l'ancienne Grèce :
Mais ce champ ne se peut tellement moissonner
Que les derniers venus n'y trouvent à glaner.
La feinte est un pays plein de terres désertes ;
Tous les jours nos auteurs y font des découvertes.
Je t'en veux dire un trait assez bien inventé :
Autrefois à Racan Malherbe l'a conté.
Ces deux rivaux d'Horace, héritiers de sa lyre,
Disciples d'Apollon, nos maîtres, pour mieux dire,
Se rencontrant un jour tout seuls et sans témoins
(Comme ils se confiaient leurs pensers et leurs soins),
Racan commence ainsi : Dites-moi, je vous prie,
Vous qui devez savoir les choses de la vie,
Qui par tous ses degrés avez déjà passé,
Et que rien ne doit fuir en cet âge avancé,
À quoi me résoudrai-je ? Il est temps que j'y pense.
Vous connaissez mon bien, mon talent, ma naissance :
Dois-je dans la province établir mon séjour,
Prendre emploi dans l'armée, ou bien charge à la cour ?
Tout au monde est mêlé d'amertume et de charmes :
La guerre a ses douceurs, l'hymen a ses alarmes.
Si je suivais mon goût, je saurais où buter ;
Mais j'ai les miens, la cour, le peuple à contenter.
Malherbe là-dessus : Contenter tout le monde !
Écoutez ce récit avant que je réponde.
J'ai lu dans quelque endroit qu'un meunier et son fils,
L'un vieillard, l'autre enfant, non pas des plus petits,
Mais garçon de quinze ans, si j'ai bonne mémoire,
Allaient vendre leur âne, un certain jour de foire.
Afin qu'il fût plus frais et de meilleur débit,
On lui lia les pieds, on vous le suspendit ;
Puis cet homme et son fils le portent comme un lustre.
Pauvres gens ! idiots ! couple ignorant et rustre !
Le premier qui les vit de rire s'éclata :
Quelle farce, dit-il, vont jouer ces gens-là ?
Le plus âne des trois n'est pas celui qu'on pense.
Le meunier, à ces mots, connaît son ignorance ;
Il met sur pieds sa bête, et la fait détaler.
L'âne, qui goûtait fort l'autre façon d'aller,
Se plaint en son patois. Le meunier n'en a cure,
Il fait monter son fils, il suit : et, d'aventure,
Passent trois bons marchands. Cet objet leur déplut.
Le plus vieux au garçon s'écria tant qu'il put :
Oh là ! oh ! descendez, que l'on ne vous le dise,
Jeune homme, qui menez laquais à barbe grise !
C'était à vous de suivre, au vieillard de monter.
Messieurs, dit le meunier, il vous faut contenter.
L'enfant met pied à terre, et puis le vieillard monte ;
Quand trois filles passant, l'une dit : C'est grand'honte
Qu'il faille voir ainsi clocher ce jeune fils,
Tandis que ce nigaud, comme un évêque assis,
Fait le veau sur son âne, et pense être bien sage.
Il n'est, dit le meunier, plus de veaux à mon âge :
Passez votre chemin, la fille, et m'en croyez.
Après maints quolibets coup sur coup renvoyés,
L'homme crut avoir tort, et mit son fils en croupe.
Au bout de trente pas, une troisième troupe
Trouve encore à gloser. L'un dit : Ces gens sont fous !
Le baudet n'en peut plus ; il mourra sous leurs coups.
Eh quoi ! charger ainsi cette pauvre bourrique !
N'ont-ils point de pitié de leur vieux domestique ?
Sans doute qu'à la foire ils vont vendre sa peau.
Parbleu ! dit le meunier, est bien fou du cerveau
Qui prétend contenter tout le monde et son père.
Essayons toutefois si par quelque manière
Nous en viendrons à bout. Ils descendent tous deux.
L'âne se prélassant marche seul devant eux.
Un quidam les rencontre et dit : Est-ce la mode
Que baudet aille à l'aise, et meunier s'incommode ?
Qui de l'âne ou du maître est fait pour se lasser ?
Je conseille à ces gens de le faire enchâsser.
Ils usent leurs souliers, et conservent leur âne !
Nicolas, au rebours : car, quand il va voir Jeanne,
Il monte sur sa bête ; et la chanson le dit.
Beau trio de baudets ! le meunier repartit :
Je suis âne, il est vrai, j'en conviens, je l'avoue ;
Mais que dorénavant on me blâme, on me loue,
Qu'on dise quelque chose ou qu'on ne dise rien,
J'en veux faire à ma tête. Il le fit, et fit bien.
Quant à vous, suivez Mars, ou l'Amour, ou le prince ;
Allez, venez, courez ; demeurez en province ;
Prenez femme, abbaye, emploi, gouvernement :
Les gens en parleront, n'en doutez nullement.


LE CHAT, LA BELETTE, ET LE PETIT LAPIN
Du palais d'un jeune lapin
Dame belette, un beau matin,
S'empara : c'est une rusée.
Le maître étant absent, ce lui fut chose aisée.
Elle porta chez lui ses pénates, un jour
Qu'il était allé faire à l'Aurore sa cour
Parmi le thym et la rosée.
Après qu'il eut brouté, trotté, fait tous ses tours,
Jeannot lapin retourne aux souterrains séjours.
La belette avait mis le nez à la fenêtre.
Ô dieux hospitaliers ! que vois-je ici paraître ?
Dit l'animal chassé du paternel logis.
Holà ! madame la belette,
Que l'on déloge sans trompette,
Ou je vais avertir tous les rats du pays.
La dame au nez pointu répondit que la terre
Était au premier occupant.
C'était un beau sujet de guerre,
Qu'un logis où lui-même il n'entrait qu'en rampant !
Et quand ce serait un royaume,
Je voudrais bien savoir, dit-elle, quelle loi
En a pour toujours fait l'octroi
À Jean, fils ou neveu de Pierre ou de Guillaume,
Plutôt qu'à Paul, plutôt qu'à moi.
Jean lapin allégua la coutume et l'usage :
Ce sont, dit-il, leurs lois qui m'ont de ce logis
Rendu maître et seigneur, et qui, de père en fils,
L'ont de Pierre à Simon, puis à moi Jean, transmis.
Le premier occupant, est-ce une loi plus sage ?
Or bien, sans crier davantage,
Rapportons-nous, dit-elle, à Raminagrobis.
C'était un chat vivant comme un dévot ermite,
Un chat faisant la chattemite,
Un saint homme de chat, bien fourré, gros et gras,
Arbitre expert sur tous les cas.
Jean lapin pour juge l'agrée.
Les voilà tous deux arrivés
Devant sa majesté fourrée.
Grippeminaud leur dit : Mes enfants, approchez,
Approchez : je suis sourd, les ans en sont la cause.
L'un et l'autre approcha, ne craignant nulle chose.
Aussitôt qu'à portée il vit les contestants,
Grippeminaud le bon apôtre,
Jetant des deux côtés la griffe en même temps,
Mit les plaideurs d'accord en croquant l'un et l'autre.
Ceci ressemble fort aux débats qu'ont parfois
Les petits souverains se rapportant aux rois.


LES DEUX MULETS
Deux mulets cheminaient, l'un d'avoine chargé,
L'autre portant l'argent de la gabelle.
Celui-ci, glorieux d'une charge si belle,
N'eût voulu pour beaucoup en être soulagé.
Il marchait d'un pas relevé
Et faisait sonner sa sonnette ;
Quand l'ennemi se présentant,
Comme il en voulait à l'argent,
Sur le mulet du fisc une troupe se jette,
Le saisit au frein, et l'arrête.
Le mulet, en se défendant,
Se sent percer de coups ; il gémit, il soupire.
Est-ce donc là, dit-il, ce qu'on m'avait promis ?
Ce mulet qui me suit du danger se retire,
Et moi j'y tombe et je péris !
Ami, lui dit son camarade,
Il n'est pas toujours bon d'avoir un haut emploi :
Si tu n'avais servi qu'un meunier comme moi,
Tu ne serais pas si malade.
"""

print(len(corpus), 'caractères')

On découpe le corpus en **documents** : une fable = un document. Le titre
d'une fable est écrit tout en majuscules, ça sert de séparateur.

In [ ]:
def decouper_en_fables(texte):
    """Un document par fable. Le titre en MAJUSCULES marque le début."""
    docs, courant = [], []
    for ligne in texte.split('\n'):
        if ligne.isupper() and len(ligne.strip()) > 4:   # une ligne de titre
            if courant:
                docs.append('\n'.join(courant).strip())
            courant = [ligne]
        else:
            courant.append(ligne)
    if courant:
        docs.append('\n'.join(courant).strip())
    return [d for d in docs if d.strip()]

fables = decouper_en_fables(corpus)
mots_moyens = sum(len(d.split()) for d in fables) / len(fables)
print(f'{len(fables)} fables | {mots_moyens:.0f} mots par fable en moyenne')

## 2. Les vraies sources francophones (avec repli hors ligne)

Un vrai corpus francophone se construit sur deux piliers publics :

- **Wikipédia FR** : `wikimedia/wikipedia`, config `20231101.fr`, licence
  CC-BY-SA 3.0. Petite, très propre, relue par des humains.
- **FineWeb-2** (sous-ensemble français `fra_Latn`) :
  `HuggingFaceFW/fineweb-2`, licence ODC-By 1.0. Un crawl du web, immense
  mais bruité.

On n'en prend qu'un **échantillon** (quelques documents en streaming), jamais
le crawl complet : le but est de montrer le pipeline de curation, pas de
télécharger des téraoctets. Et si `datasets` manque ou si le réseau est
coupé, on retombe sur les fables. Le pipeline est identique quelle que soit
la source.

In [ ]:
def charger_echantillon_web(n=8):
    """Renvoie n petits documents web francophones, ou un repli sur les fables.

    Toute la cellule est gardée : elle ne lève jamais, même sans réseau.
    On épingle la révision du dataset pour la reproductibilité (un livre imprimé
    ne bouge pas).
    """
    if HAS_DATASETS:
        try:
            from datasets import load_dataset
            ds = load_dataset(
                'HuggingFaceFW/fineweb-2',
                name='fra_Latn',
                split='train',
                revision='af9c133',   # révision épinglée
                streaming=True,
            )
            textes = [ex['text'] for _, ex in zip(range(n), ds)]
            if textes:
                return textes, 'FineWeb-2 (fra_Latn)'
        except Exception as e:
            print('  [repli] FineWeb-2 indisponible :', type(e).__name__)
    # repli hors ligne : des extraits de fables tiennent lieu d'echantillon web
    return fables[:n], 'fables (repli hors ligne)'


echantillon, source = charger_echantillon_web(n=8)
print(f'{len(echantillon)} documents chargés depuis : {source}')
print('premier document (100 caractères) :', repr(echantillon[0][:100]))

<div style="border-left:4px solid #8FA877;padding-left:12px">

**Comme un vrai labo.** Ici on manipule 8 documents. OLMo 2 (AI2) est
pré-entraîné sur **OLMo-mix-1124** (~3 900 milliards de tokens), puis affiné
sur **Dolmino-mix-1124** (843 milliards) ; le corpus **Dolma** (`allenai/dolma`, ODC-By) est celui d'OLMo 1. Le geste est le même, l'échelle change de
douze ordres de grandeur. Le pipeline que tu écris ici, c'est celui de Dolma
en miniature.

</div>

## 3. On salit le corpus

Pour avoir quelque chose à nettoyer, on fabrique un **corpus brut** réaliste :
les 30 fables propres, plus des doublons exacts, des quasi-doublons (un
en-tête de site ajouté), et quelques documents de junk. C'est ce corpus sale
qui entre dans le pipeline.

In [ ]:
brut = list(fables)                                    # les 30 fables propres

# doublons EXACTS : 6 fables copiées à l'identique (sites miroirs)
brut += fables[:6]

# QUASI-doublons : 5 fables avec un en-tête et un pied de page de site
for f in fables[:5]:
    brut.append('[Accueil] [Connexion] [Panier] ' + f + '\nTous droits réservés 2024.')

# JUNK : 4 documents sans valeur
junk = [
    'cliquez ici ' * 8,                                # répétition pure
    '404 not found',                                   # trop court
    '>>> === ||| ### @@@ *** ((( ))) {{{ }}} +++ --- /// %%% $$$ !!! ???',  # ponctuation
    'achat vente promo solde discount deal offre bon plan code reduction cashback',  # sans mots-outils
]
brut += junk

random.shuffle(brut)
print(f'corpus brut : {len(brut)} documents (30 propres + 6 exacts + 5 quasi + 4 junk)')

## 4. Déduplication exacte (hash)

Le cas facile : retirer les copies **octet pour octet**. On calcule une
empreinte (un hash SHA256) par document, et on garde les empreintes déjà vues
dans un `set`. Deux documents identiques donnent la même empreinte ; un seul
octet de différence change tout le hash.

In [ ]:
def hash_doc(texte):
    """Empreinte SHA256 : 64 caractères hexadécimaux, unique par contenu."""
    return hashlib.sha256(texte.encode('utf-8')).hexdigest()


def dedup_exacte(documents):
    vus, gardes = set(), []
    for doc in documents:
        h = hash_doc(doc)
        if h not in vus:        # jamais vu -> on garde
            vus.add(h)
            gardes.append(doc)
        # déjà vu -> doublon exact, on jette
    return gardes


apres_exacte = dedup_exacte(brut)
print(f'{len(brut)} -> {len(apres_exacte)} documents  '
      f'(-{len(brut) - len(apres_exacte)} doublons exacts)')

In [ ]:
assert len(apres_exacte) == len(brut) - 6, (
    f'attendu {len(brut) - 6} après dédup exacte, obtenu {len(apres_exacte)} : '
    'les 6 copies à l\'identique doivent partir')
print('dédup exacte validée : les 6 doublons exacts ont été retirés')

## 5. Quasi-doublons : shingles, MinHash, LSH

Le hash exact ne voit rien des quasi-doublons : ajoute `[Accueil]` en tête et
le hash change du tout au tout. Il faut une mesure de **ressemblance**. On
représente chaque document par son ensemble de **shingles** (des séquences de
3 mots qui se chevauchent), puis on compare ces ensembles avec la similarité
de **Jaccard** : la taille de l'intersection sur la taille de l'union.

In [ ]:
def shingles(texte, n=3):
    """Ensemble des séquences de n mots consécutifs (fenêtre glissante)."""
    texte = texte.lower()
    texte = re.sub(r'[^0-9a-zàâäéèêëîïôöùûüç\s]', ' ', texte)
    mots = texte.split()
    ens = {' '.join(mots[i:i + n]) for i in range(len(mots) - n + 1)}
    return ens or {texte}


def jaccard(a, b):
    A, B = shingles(a), shingles(b)
    union = len(A | B)
    return len(A & B) / union if union else 0.0


# une fable et sa version avec en-tête de site : quasi identiques
originale = fables[0]
avec_entete = '[Accueil] [Connexion] [Panier] ' + fables[0] + '\nTous droits réservés 2024.'
print(f'Jaccard(originale, quasi-doublon) = {jaccard(originale, avec_entete):.3f}')
print(f'Jaccard(fable 0, fable 1)         = {jaccard(fables[0], fables[1]):.3f}')

### La signature MinHash, en intuition

Comparer les ensembles complets coûte cher. **MinHash** résume chaque
document en quelques nombres : pour K fonctions de hash, on garde le plus
petit hash obtenu sur les shingles. Le théorème de Broder dit que la fraction
de positions où deux signatures coïncident **estime** la Jaccard. On compare
des signatures courtes au lieu des documents entiers.

In [ ]:
def signature_minhash(sh, K=64):
    """K min-hashes : la signature du document. Forme : (K,)."""
    return [min(hash((s, k)) for s in sh) for k in range(K)]


def jaccard_estimee(sig1, sig2):
    return sum(a == b for a, b in zip(sig1, sig2)) / len(sig1)


s_orig = signature_minhash(shingles(originale))
s_quasi = signature_minhash(shingles(avec_entete))
print(f'Jaccard exacte : {jaccard(originale, avec_entete):.3f}')
print(f'Jaccard MinHash: {jaccard_estimee(s_orig, s_quasi):.3f}  '
      f'(estimée sur K=64 hashes)')

### LSH : ne comparer que les bons candidats

On ne veut pas comparer tous les documents deux à deux. Le **LSH** découpe
chaque signature en bandes ; deux documents partageant une bande entière
tombent dans le même seau et deviennent **candidats**. On ne vérifie la vraie
Jaccard que sur ces candidats.

In [ ]:
def dedup_minhash(documents, K=64, bandes=16, seuil=0.7):
    lignes = K // bandes
    sigs = [signature_minhash(shingles(d), K) for d in documents]

    # LSH : ranger chaque doc dans un seau par bande
    seaux = defaultdict(list)
    for i, sig in enumerate(sigs):
        for b in range(bandes):
            bande = tuple(sig[b * lignes:(b + 1) * lignes])
            seaux[(b, hash(bande))].append(i)

    # paires candidates : deux docs dans un même seau
    candidats = set()
    for ids in seaux.values():
        for a in range(len(ids)):
            for c in range(a + 1, len(ids)):
                candidats.add((ids[a], ids[c]))

    # vérification exacte : on retire un doc par paire trop proche
    a_retirer = set()
    for i, j in candidats:
        if i in a_retirer or j in a_retirer:
            continue
        if jaccard(documents[i], documents[j]) > seuil:
            a_retirer.add(j)
    gardes = [d for k, d in enumerate(documents) if k not in a_retirer]
    return gardes, len(candidats)


apres_minhash, n_candidats = dedup_minhash(apres_exacte)
print(f'{len(apres_exacte)} -> {len(apres_minhash)} documents  '
      f'(-{len(apres_exacte) - len(apres_minhash)} quasi-doublons, '
      f'{n_candidats} paires candidates testées)')

In [ ]:
assert len(apres_minhash) == len(apres_exacte) - 5, (
    f'attendu {len(apres_exacte) - 5} après MinHash, obtenu {len(apres_minhash)} : '
    'les 5 quasi-doublons doivent partir')
print('MinHash + LSH validé : les 5 quasi-doublons ont été retirés')

## 6. Filtres de qualité (règles type Gopher, simplifiées)

La dédup a retiré les répétitions. Reste le **junk** : des documents uniques
mais sans valeur. On les jette avec des **heuristiques** bon marché, inspirées
des règles du modèle Gopher (DeepMind). On teste chaque règle, on accumule les
**raisons** de rejet, et un document ne passe que si aucune règle ne le recale.

In [ ]:
MOTS_OUTILS = {
    'le', 'la', 'les', 'de', 'des', 'du', 'un', 'une', 'et', 'à', 'en',
    'que', 'qui', 'dans', 'pour', 'sur', 'au', 'aux', 'ce', 'il', 'elle',
    'se', 'ne', 'pas', 'son', 'sa', 'est',
}
PONCTUATION = set('.,;:!?-«»"\'()[]{}/\\|@#*+=%$')


def statistiques(texte):
    mots = texte.split()
    nb_mots = len(mots)
    ratio_ponct = sum(c in PONCTUATION for c in texte) / max(len(texte), 1)
    plus_frequent = Counter(m.lower() for m in mots).most_common(1)
    ratio_repetition = plus_frequent[0][1] / nb_mots if nb_mots else 1.0
    ratio_outils = sum(m.lower() in MOTS_OUTILS for m in mots) / nb_mots if nb_mots else 0.0
    return nb_mots, ratio_ponct, ratio_repetition, ratio_outils

In [ ]:
def filtre_qualite(texte, min_mots=12, max_ponct=0.30,
                   max_repetition=0.20, min_outils=0.10):
    """Renvoie (gardé ?, liste des raisons de rejet)."""
    nb_mots, ratio_ponct, ratio_rep, ratio_outils = statistiques(texte)
    raisons = []
    if nb_mots < min_mots:
        raisons.append('trop_court')
    if ratio_ponct > max_ponct:
        raisons.append('trop_de_ponctuation')
    if ratio_rep > max_repetition:
        raisons.append('trop_repetitif')
    if ratio_outils < min_outils:
        raisons.append('trop_peu_de_mots_outils')
    return len(raisons) == 0, raisons

In [ ]:
gardes, raisons_rejet = [], Counter()
for doc in apres_minhash:
    ok, raisons = filtre_qualite(doc)
    if ok:
        gardes.append(doc)
    else:
        for r in raisons:
            raisons_rejet[r] += 1

print(f'{len(apres_minhash)} -> {len(gardes)} documents  '
      f'(-{len(apres_minhash) - len(gardes)} junk)')
print('raisons de rejet :', dict(raisons_rejet))

assert len(gardes) == 30, f'attendu 30 fables propres, obtenu {len(gardes)}'
print('filtre qualité validé : on retrouve les 30 fables propres')

### L'entonnoir, chiffré

On récapitule les survivants à chaque étape. C'est le **rapport** que regarde
un labo pour vérifier que son pipeline fait ce qu'il croit.

In [ ]:
etapes = [
    ('Corpus brut', len(brut)),
    ('Après dédup exacte', len(apres_exacte)),
    ('Après MinHash + LSH', len(apres_minhash)),
    ('Après filtres qualité', len(gardes)),
]
for nom, n in etapes:
    barre = '#' * n
    print(f'{nom:24s} {n:3d}  {barre}')
print(f'\nbilan : {len(brut)} -> {len(gardes)} documents '
      f'({100 * (len(brut) - len(gardes)) / len(brut):.0f}% jetés)')

## 7. Le cas qui échoue : entraîner sur des doublons massifs

Pourquoi tout ce travail ? Parce que les doublons **abîment le modèle**, pas
seulement le disque. On le montre chiffres à l'appui. On entraîne deux fois
le même petit modèle de caractères sur le **même budget** de tokens :

- une fois sur 24 fables **distinctes** (corpus propre) ;
- une fois sur un corpus où **3 fables sont copiées 20 fois** (corpus
  dominé par des doublons).

On mesure la loss (perte, erreur) sur 6 fables **jamais vues**. Le corpus
doublonné colle mieux à son train mais généralise moins bien : c'est la
mémorisation contre la généralisation, exactement ce que Lee et al. (2022)
ont mesuré à grande échelle. Tout est dans un `try/except` : même si torch
manque, le notebook ne casse pas.

In [ ]:
try:
    import torch
    import torch.nn as nn

    random.seed(42)
    tous = decouper_en_fables(corpus)
    random.shuffle(tous)
    train_docs, eval_docs = tous[:24], tous[24:]

    chars = sorted(set(corpus))
    stoi = {c: i for i, c in enumerate(chars)}
    V = len(chars)
    bloc = 48

    def encoder(t):
        return [stoi[c] for c in t if c in stoi]

    def en_tokens(liste_docs):
        return torch.tensor(encoder('\n'.join(liste_docs)))

    class PetitModele(nn.Module):
        """Minuscule : embedding + moyenne causale + MLP. Rapide sur CPU."""
        def __init__(self, V, d=48, cache=64):
            super().__init__()
            self.emb = nn.Embedding(V, d)
            self.pos = nn.Embedding(bloc, d)
            self.tete = nn.Sequential(nn.Linear(d, cache), nn.Tanh(), nn.Linear(cache, V))
        def forward(self, x):
            B, T = x.shape
            e = self.emb(x) + self.pos(torch.arange(T))
            contexte = torch.cumsum(e, 1) / torch.arange(1, T + 1).view(1, T, 1)
            return self.tete(contexte)

    def batch(data, taille=48):
        ix = torch.randint(0, len(data) - bloc - 1, (taille,))
        x = torch.stack([data[i:i + bloc] for i in ix])
        y = torch.stack([data[i + 1:i + bloc + 1] for i in ix])
        return x, y

    def loss_moyenne(m, data, n=60):
        m.eval()
        with torch.no_grad():
            vals = []
            for _ in range(n):
                x, y = batch(data)
                logits = m(x)
                vals.append(nn.functional.cross_entropy(logits.view(-1, V), y.view(-1)).item())
        return sum(vals) / len(vals)

    def entrainer(data, pas=800):
        torch.manual_seed(42)
        m = PetitModele(V)
        opt = torch.optim.AdamW(m.parameters(), lr=5e-3)
        for _ in range(pas):
            x, y = batch(data)
            logits = m(x)
            perte = nn.functional.cross_entropy(logits.view(-1, V), y.view(-1))
            opt.zero_grad(); perte.backward(); opt.step()
        return m

    eval_data = en_tokens(eval_docs)

    # corpus PROPRE : 24 fables distinctes
    propre = en_tokens(train_docs)
    m_propre = entrainer(propre)
    tr_propre, ev_propre = loss_moyenne(m_propre, propre), loss_moyenne(m_propre, eval_data)

    # corpus DOUBLONNÉ : + 3 fables copiées 20 fois
    doublonne = en_tokens(train_docs + train_docs[:3] * 20)
    m_doublon = entrainer(doublonne)
    tr_doublon, ev_doublon = loss_moyenne(m_doublon, doublonne), loss_moyenne(m_doublon, eval_data)

    print(f'CORPUS PROPRE    : train={tr_propre:.3f}  eval={ev_propre:.3f}  '
          f'écart={ev_propre - tr_propre:.3f}')
    print(f'CORPUS DOUBLONNÉ : train={tr_doublon:.3f}  eval={ev_doublon:.3f}  '
          f'écart={ev_doublon - tr_doublon:.3f}')
    print()
    print(f'les doublons creusent l\'écart train/eval de '
          f'{ev_propre - tr_propre:.2f} à {ev_doublon - tr_doublon:.2f} : '
          f'le modèle mémorise au lieu de généraliser')
except Exception as e:
    print('[entraînement sauté]', type(e).__name__, ':', e)

### Un second piège : le filtre trop gourmand

L'erreur inverse est aussi coûteuse. Un filtre trop sévère ne nettoie pas, il
**vide** le corpus. On exige ici moins de 5 % de ponctuation et plus de 35 %
de mots-outils : aucune prose française n'y survit.

In [ ]:
def filtre_trop_agressif(texte):
    nb_mots, ratio_ponct, ratio_rep, ratio_outils = statistiques(texte)
    # seuils absurdes pour de la prose
    return nb_mots >= 12 and ratio_ponct < 0.05 and ratio_rep < 0.20 and ratio_outils >= 0.35


survivants = sum(filtre_trop_agressif(d) for d in fables)
print(f'filtre normal        : {sum(filtre_qualite(d)[0] for d in fables)}/30 fables gardées')
print(f'filtre trop agressif : {survivants}/30 fables gardées  -> corpus vidé')
assert survivants == 0, 'le filtre absurde devrait tout jeter'
print('leçon : un filtre qu\'on ne mesure pas jette en silence ce qu\'on voulait garder')

## 8. Mélanger les sources et upsampler

Propre ne veut pas dire **équilibré**. Le web écrase tout par le volume. On
choisit donc les **proportions** du mélange final, et on **upsample** (voir
plus souvent) les sources propres et précieuses. Ici on donne trois poids
et on calcule combien de fois chaque source sera vue.

In [ ]:
sources = {
    'fineweb_fr': {'documents': 8000, 'poids': 1.0},   # web, vu 1 fois
    'wikipedia_fr': {'documents': 1500, 'poids': 2.0}, # propre, vu 2 fois
    'fables': {'documents': 30, 'poids': 5.0},         # rare et soigné, vu 5 fois
}


def parts_du_melange(sources):
    """Part effective de chaque source = (documents x poids) / masse totale."""
    masse = {n: s['documents'] * s['poids'] for n, s in sources.items()}
    total = sum(masse.values())
    return {n: m / total for n, m in masse.items()}


parts = parts_du_melange(sources)
for nom, part in parts.items():
    brut_pct = sources[nom]['documents'] / sum(s['documents'] for s in sources.values())
    print(f"{nom:14s} : {brut_pct * 100:5.1f}% brut -> {part * 100:5.1f}% après upsampling")

In [ ]:
part_fables = parts['fables']
part_fables_brut = 30 / (8000 + 1500 + 30)
assert part_fables > part_fables_brut, 'l\'upsampling doit augmenter la part des fables'
print(f'\nles fables passent de {part_fables_brut * 100:.2f}% (volume brut) '
      f'à {part_fables * 100:.2f}% (après upsampling ×5)')
print('la composition du corpus, pas son volume brut, décide des forces du modèle')

## Exercices

À toi de jouer : quatre exercices, du plus simple (●) au plus costaud (●●●),
sur les gestes du chapitre. Chaque cellule marquée `# TODO(toi)` contient un
trou ; complète-le, puis exécute la cellule de validation (`assert`) qui
suit : si elle passe sans erreur, c'est gagné.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à
ta place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est
toi. Les réponses sont dans le notebook solution, à n'ouvrir qu'après avoir
vraiment essayé.

### Exercice 1 · Déduplication exacte par hash — niveau ●

Refais le geste de la section 4 sans la regarder. Garde un document seulement
si son empreinte SHA256 n'a jamais été vue : ajoute alors le hash à `vus` et
le document à `gardes`. Rappel : un seul octet de différence change tout le
hash, la variante avec un « ! » doit donc survivre.

In [ ]:
def dedup_exacte(documents):
    vus, gardes = set(), []
    for doc in documents:
        h = hashlib.sha256(doc.encode('utf-8')).hexdigest()
        # TODO(toi) : garde le doc seulement si son hash n'a jamais été vu.
        # Ajoute le hash à `vus` et le doc à `gardes` dans ce cas.
        ...
    return gardes


miroirs = [
    'Maître Corbeau, sur un arbre perché, tenait en son bec un fromage.',
    'La cigale, ayant chanté tout l\'été, se trouva fort dépourvue.',
    'Maître Corbeau, sur un arbre perché, tenait en son bec un fromage.',
    'Maître Corbeau, sur un arbre perché, tenait en son bec un fromage !',
    'La cigale, ayant chanté tout l\'été, se trouva fort dépourvue.',
]
uniques = dedup_exacte(miroirs)
print(f'{len(miroirs)} -> {len(uniques)} documents')

In [ ]:
# Validation : déduplication exacte.
assert len(uniques) == 3, f'attendu 3 documents uniques, obtenu {len(uniques)}'
assert uniques[0].startswith('Maître Corbeau'), 'l\'ordre de première apparition doit être conservé'
assert uniques[2].endswith('!'), 'un octet de différence (le « ! ») doit suffire à garder le document'
print('Dédup exacte OK : 5 -> 3, les copies parfaites partent, la variante à un octet reste')

### Exercice 2 · La règle des mots-outils — niveau ●

Complète le filtre qualité de la section 6 : rejette un document, avec la
raison `'trop_peu_de_mots_outils'`, si son ratio de mots-outils est **sous**
le seuil `min_outils`. Un texte humain contient une proportion stable de
petits mots (« le », « de », « et »…) ; une liste de mots-clés n'en a
presque pas.

In [ ]:
def filtre_qualite(texte, min_mots=12, max_ponct=0.30,
                   max_repetition=0.20, min_outils=0.10):
    """Renvoie (gardé ?, liste des raisons de rejet)."""
    nb_mots, ratio_ponct, ratio_rep, ratio_outils = statistiques(texte)
    raisons = []
    if nb_mots < min_mots:
        raisons.append('trop_court')
    if ratio_ponct > max_ponct:
        raisons.append('trop_de_ponctuation')
    if ratio_rep > max_repetition:
        raisons.append('trop_repetitif')
    # TODO(toi) : rejette (raison 'trop_peu_de_mots_outils') si le ratio de
    # mots-outils est SOUS le seuil min_outils.
    ...
    return len(raisons) == 0, raisons


prose = 'La cigale, ayant chanté tout l\'été, se trouva fort dépourvue quand la bise fut venue.'
spam = 'achat vente promo solde discount deal offre bon plan code reduction cashback'
print('prose :', filtre_qualite(prose))
print('spam  :', filtre_qualite(spam))

In [ ]:
# Validation : règle des mots-outils.
ok_prose, raisons_prose = filtre_qualite(prose)
ok_spam, raisons_spam = filtre_qualite(spam)
assert ok_prose and raisons_prose == [], f'la prose devrait passer, raisons : {raisons_prose}'
assert not ok_spam, 'la liste de mots-clés devrait être rejetée'
assert raisons_spam == ['trop_peu_de_mots_outils'], f'raison attendue unique, obtenu : {raisons_spam}'
print('Filtre OK : la prose passe, la liste de mots-clés est rejetée pour', raisons_spam[0])

### Exercice 3 · La part effective du mélange — niveau ●●

Reprends le mélange de la section 8 : calcule la masse (documents × poids) de
chaque source, puis divise par la masse totale pour obtenir la part effective
de chacune. L'upsampling doit augmenter la part des fables au-delà de leur
volume brut.

In [ ]:
sources = {
    'fineweb_fr': {'documents': 8000, 'poids': 1.0},   # web, vu 1 fois
    'wikipedia_fr': {'documents': 1500, 'poids': 2.0}, # propre, vu 2 fois
    'fables': {'documents': 30, 'poids': 5.0},         # rare et soigné, vu 5 fois
}


def parts_du_melange(sources):
    """Part effective de chaque source = (documents x poids) / masse totale."""
    # TODO(toi) : calcule la masse (documents x poids) de chaque source,
    # puis divise par la masse totale pour obtenir la part de chacune.
    masse = ...
    total = ...
    return {n: m / total for n, m in masse.items()}


parts = parts_du_melange(sources)
for nom, part in parts.items():
    print(f'{nom:14s} : {part * 100:5.1f}% du mélange')

In [ ]:
# Validation : mélange et upsampling.
part_fables = parts['fables']
part_fables_brut = 30 / (8000 + 1500 + 30)
assert abs(sum(parts.values()) - 1.0) < 1e-9, 'les parts doivent sommer à 1'
assert part_fables > part_fables_brut, 'l\'upsampling doit augmenter la part des fables'
assert abs(parts['wikipedia_fr'] - 3000 / 11150) < 1e-9, (
    f"part wikipedia attendue 26.9%, obtenu {parts['wikipedia_fr'] * 100:.1f}%")
print(f'Mélange OK : les fables passent de {part_fables_brut * 100:.2f}% (volume brut) '
      f'à {part_fables * 100:.2f}% (après upsampling ×5)')

### Exercice 4 · La vérification Jaccard du LSH — niveau ●●●

Le LSH de la section 5 ne fait que **proposer** des paires candidates.
Complète la vérification finale de `dedup_minhash` : pour chaque paire
candidate `(i, j)`, si la vraie Jaccard des deux documents dépasse `seuil`,
marque `j` comme doublon à retirer. LSH propose, Jaccard dispose.

In [ ]:
def dedup_minhash(documents, K=64, bandes=16, seuil=0.7):
    lignes = K // bandes
    sigs = [signature_minhash(shingles(d), K) for d in documents]

    seaux = defaultdict(list)
    for i, sig in enumerate(sigs):
        for b in range(bandes):
            bande = tuple(sig[b * lignes:(b + 1) * lignes])
            seaux[(b, hash(bande))].append(i)

    candidats = set()
    for ids in seaux.values():
        for a in range(len(ids)):
            for c in range(a + 1, len(ids)):
                candidats.add((ids[a], ids[c]))

    a_retirer = set()
    for i, j in candidats:
        if i in a_retirer or j in a_retirer:
            continue
        # TODO(toi) : si la vraie Jaccard des docs i et j dépasse `seuil`,
        # marque j comme doublon à retirer.
        ...
    gardes = [d for k, d in enumerate(documents) if k not in a_retirer]
    return gardes, len(candidats)


originale = ('Rien ne sert de courir, il faut partir à point : le lièvre et la tortue '
             'en sont un témoignage que la fable raconte à tous les enfants.')
quasi = '[Accueil] ' + originale
autres = [
    'La raison du plus fort est toujours la meilleure, nous l\'allons montrer tout à l\'heure.',
    'Tout flatteur vit aux dépens de celui qui l\'écoute, cette leçon vaut bien un fromage sans doute.',
]
petits = [originale, quasi] + autres
gardes_lsh, n_cand = dedup_minhash(petits)
print(f'{len(petits)} -> {len(gardes_lsh)} documents ({n_cand} paire(s) candidate(s) testée(s))')

In [ ]:
# Validation : LSH + vérification Jaccard.
assert len(gardes_lsh) == 3, f'attendu 3 documents, obtenu {len(gardes_lsh)} : le quasi-doublon doit partir'
assert originale in gardes_lsh, 'l\'originale doit rester'
assert quasi not in gardes_lsh, 'la copie avec en-tête doit être retirée'
print('MinHash + LSH OK : le quasi-doublon est parti, les documents distincts restent')

## Ce que tu as construit

Un pipeline de données complet, de tes mains, hors ligne : chargement (avec
repli), déduplication exacte par hash, MinHash + LSH pour les quasi-doublons,
filtres de qualité type Gopher, mélange et upsampling des sources. Et surtout,
la preuve chiffrée que les doublons abîment la généralisation.

Tu as maintenant de quoi nourrir sérieusement le GPT du chapitre 10. Au
**chapitre 14**, on passe à la question du coût : combien de paramètres,
combien de mémoire, combien de temps ? On apprend à compter avant de lancer.